#### Project Overview & Business Questions

This project explores customer transaction data to understand purchasing behavior and identify patterns associated with customer churn.

The analysis focuses on customer activity, spending behavior, cart abandonment, membership duration, and feedback.

**Business Questions**
- How is customer churn distributed across the dataset?
- How does purchasing behavior differ between churned and non-churned customers?
- Is cart abandonment associated with customer churn?
- How do churn rates vary across countries?
- What insights can customer feedback provide, and what are the limitations?

In [1]:
print('Hello Ahmed')

Hello Ahmed


In [57]:
### 02. Data Loading & Initial Assessment

###  Data Loading & Initial Assessment

In [45]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly as py

In [3]:
import os

print(os.getcwd())

e:\Data Scientest Diploma\New Group\Mid project\Customer_Transactions_Project


In [4]:
df = pd.read_csv('E:\\Data Scientest Diploma\\New Group\\Mid project\\Customer_Transactions_Project\\Customer_Transactions.csv')
df.head()   
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   customer_id               10000 non-null  int64  
 1   age                       10000 non-null  int64  
 2   gender                    10000 non-null  str    
 3   country                   10000 non-null  str    
 4   annual_income             10000 non-null  int64  
 5   spending_score            10000 non-null  int64  
 6   num_purchases             10000 non-null  int64  
 7   avg_purchase_value        10000 non-null  float64
 8   membership_years          10000 non-null  int64  
 9   website_visits_per_month  10000 non-null  int64  
 10  cart_abandon_rate         10000 non-null  float64
 11  churned                   10000 non-null  int64  
 12  feedback_text             10000 non-null  str    
 13  last_purchase_date        10000 non-null  str    
dtypes: float64(2), int

### Data Quality & Preparation

In [5]:
df.isnull().sum()

customer_id                 0
age                         0
gender                      0
country                     0
annual_income               0
spending_score              0
num_purchases               0
avg_purchase_value          0
membership_years            0
website_visits_per_month    0
cart_abandon_rate           0
churned                     0
feedback_text               0
last_purchase_date          0
dtype: int64

In [6]:
df['customer_id'].nunique()

10000

In [7]:
df['customer_id'].duplicated().sum()

np.int64(0)

In [8]:
df[['annual_income','age', 'spending_score', 'num_purchases', 'avg_purchase_value', 'cart_abandon_rate']].describe().round(2)

,annual_income,age,spending_score,num_purchases,avg_purchase_value,cart_abandon_rate
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,86067.68,44.05,50.92,22.58,47.45,0.50
std,38986.79,15.40,28.75,10.16,11.21,0.29
min,20028.00,18.00,1.00,1.00,16.75,0.00
25%,55345.50,31.00,26.00,14.00,39.50,0.25
50%,78339.50,44.00,51.00,22.00,46.99,0.51
75%,115570.50,57.00,75.00,31.00,55.08,0.75
max,179960.00,70.00,100.00,49.00,83.27,1.00


In [9]:
colms = ['annual_income','age', 'spending_score', 'num_purchases', 'avg_purchase_value', 'cart_abandon_rate']

In [10]:
Q1 = df[colms].quantile(0.25)
Q3 = df[colms].quantile(0.75)
IQR = Q3 - Q1
print(Q1)
print(Q3)
print(IQR)

annual_income         55345.50
age                      31.00
spending_score           26.00
num_purchases            14.00
avg_purchase_value       39.50
cart_abandon_rate         0.25
Name: 0.25, dtype: float64
annual_income         115570.50
age                       57.00
spending_score            75.00
num_purchases             31.00
avg_purchase_value        55.08
cart_abandon_rate          0.75
Name: 0.75, dtype: float64
annual_income         60225.00
age                      26.00
spending_score           49.00
num_purchases            17.00
avg_purchase_value       15.58
cart_abandon_rate         0.50
dtype: float64


In [11]:
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = ((df[colms] < lower_bound) | (df[colms] > upper_bound)).sum()
print(outliers)

annual_income          0
age                    0
spending_score         0
num_purchases          0
avg_purchase_value    17
cart_abandon_rate      0
dtype: int64


### Exploratory Data Analysis (EDA)

In [12]:
df['churned'].value_counts()

churned
0    8910
1    1090
Name: count, dtype: int64

In [13]:
df["churned"].value_counts(normalize=True).mul(100).round(2)

churned
0    89.1
1    10.9
Name: proportion, dtype: float64

In [14]:
df.groupby("churned")[
    ["annual_income", "num_purchases", "avg_purchase_value"]
].mean().round(2)

,annual_income,num_purchases,avg_purchase_value
churned,,,
0,86119.56,23.19,47.96
1,85643.60,17.53,43.28


In [15]:
df[['churned', 'feedback_text']].head(10)

,churned,feedback_text
0,0,Very satisfied with my purchase.
1,0,Good quality and value for money.
2,0,Excellent customer service.
3,0,Good quality and value for money.
4,0,Excellent customer service.
5,1,The product did not match the description.
6,0,Love the products and fast shipping.
7,0,The website is easy to use.
8,0,Excellent customer service.
9,0,Excellent customer service.


In [16]:
df["feedback_text"].value_counts().head(10)

feedback_text
Love the products and fast shipping.          1542
Great experience, will shop again!            1510
Very satisfied with my purchase.              1507
Excellent customer service.                   1491
The website is easy to use.                   1437
Good quality and value for money.             1423
Poor quality, item broke after a week.         200
Customer service was unhelpful.                187
Shipping was too slow.                         183
The product did not match the description.     181
Name: count, dtype: int64

In [17]:
negative_feedback = [
    "Poor quality, item broke after a week.",
    "Customer service was unhelpful.",
    "Shipping was too slow.",
    "The product did not match the description."
    "I will not be shopping here again."

]

df[df["feedback_text"].isin(negative_feedback)].groupby("churned").count()

    



,customer_id,age,gender,country,annual_income,spending_score,num_purchases,avg_purchase_value,membership_years,website_visits_per_month,cart_abandon_rate,feedback_text,last_purchase_date
churned,,,,,,,,,,,,,
1,570,570,570,570,570,570,570,570,570,570,570,570,570


In [18]:
df[df["churned"] == 1]["feedback_text"].value_counts()

feedback_text
Poor quality, item broke after a week.        200
Customer service was unhelpful.               187
Shipping was too slow.                        183
The product did not match the description.    181
Difficult to navigate the website.            174
I will not be shopping here again.            165
Name: count, dtype: int64

In [19]:
df[df["churned"] == 0]["feedback_text"].value_counts()

feedback_text
Love the products and fast shipping.    1542
Great experience, will shop again!      1510
Very satisfied with my purchase.        1507
Excellent customer service.             1491
The website is easy to use.             1437
Good quality and value for money.       1423
Name: count, dtype: int64

### Insight 1 — Customer Feedback & Churn
من النتائج اللي وصلنالها لحد دلوقتي:
- إجمالي العملاء: 10,000
- العملاء اللي عندهم churned = 1: 1,090 (10.9%)
- العملاء اللي عندهم churned = 0: 8,910 (89.1%)
-كل الـ 1,090 عميل اللي عملوا Churn عندهم تعليقات سلبية، موزعة على 6 أنواع من الشكاوى
الاستنتاج المبدئي: في مجموعة البيانات دي، فيه ارتباط واضح بين سلبيةـ Feedback وChurn.


In [21]:
df.groupby("churned")[["annual_income", "num_purchases", "avg_purchase_value", "website_visits_per_month", "cart_abandon_rate"]].mean().round(2)
        
        
        
        
    

,annual_income,num_purchases,avg_purchase_value,website_visits_per_month,cart_abandon_rate
churned,,,,,
0,86119.56,23.19,47.96,15.96,0.49
1,85643.60,17.53,43.28,12.49,0.63


### العملاء اللي عملوا Churn مرتبطين بمتوسط مشتريات اقل، وزيارات اقل للموقع، ومعدل ترك سلة اعلى. دي علاقات وصف ومش دليل على السبب، ولسه محتاجين

In [26]:
df.groupby("churned")[[ "age", "membership_years", "spending_score" ]].mean().round(2)


,age,membership_years,spending_score
churned,,,
0,44.20,6.52,53.38
1,42.78,4.87,30.82


In [30]:
df.groupby("country")["churned"].mean().mul(100).round(2).sort_values (ascending=False)

country
UK              12.83
France          12.03
Japan           11.72
South Africa    11.68
Germany         11.21
Brazil          11.02
India           10.89
USA             10.05
Australia        9.80
Canada           9.66
Name: churned, dtype: float64

In [43]:
count = df[["country" , 'churned']].value_counts() 
print(count)

country       churned
USA           0          2149
India         0          1358
Brazil        0           912
Germany       0           895
UK            0           883
Canada        0           748
Australia     0           617
France        0           563
Japan         0           437
South Africa  0           348
USA           1           240
India         1           166
UK            1           130
Germany       1           113
Brazil        1           113
Canada        1            80
France        1            77
Australia     1            67
Japan         1            58
South Africa  1            46
Name: count, dtype: int64


In [42]:
count / 10000


country       churned
USA           0          0.2149
India         0          0.1358
Brazil        0          0.0912
Germany       0          0.0895
UK            0          0.0883
Canada        0          0.0748
Australia     0          0.0617
France        0          0.0563
Japan         0          0.0437
South Africa  0          0.0348
USA           1          0.0240
India         1          0.0166
UK            1          0.0130
Germany       1          0.0113
Brazil        1          0.0113
Canada        1          0.0080
France        1          0.0077
Australia     1          0.0067
Japan         1          0.0058
South Africa  1          0.0046
Name: count, dtype: float64

### Visualization 

In [46]:
import plotly.express as px

In [47]:
fig = px.histogram(
    df,
    x="churned",
    color="churned",
    title="Customer Churn Distribution",
    labels={"churned": "Customer Churn Status"},
    text_auto=True
)

fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=[0, 1],
        ticktext=["Not Churned", "Churned"]
    ),
    yaxis_title="Number of Customers",
    showlegend=False
)

fig.show()

In [48]:
purchase_comparison = df.groupby("churned")["num_purchases"].mean().reset_index()

purchase_comparison["churned"] = purchase_comparison["churned"].map({
    0: "Not Churned",
    1: "Churned"
})

fig = px.bar(
    purchase_comparison,
    x="churned",
    y="num_purchases",
    color="churned",
    text_auto=".2f",
    title="Average Number of Purchases by Churn Status",
    labels={
        "churned": "Customer Status",
        "num_purchases": "Average Number of Purchases"
    }
)

fig.show()

In [49]:
cart_comparison = df.groupby("churned")["cart_abandon_rate"].mean().reset_index()

cart_comparison["churned"] = cart_comparison["churned"].map({
    0: "Not Churned",
    1: "Churned"
})

fig = px.bar(
    cart_comparison,
    x="churned",
    y="cart_abandon_rate",
    color="churned",
    text_auto=".2f",
    title="Average Cart Abandonment Rate by Churn Status",
    labels={
        "churned": "Customer Status",
        "cart_abandon_rate": "Average Cart Abandonment Rate"
    }
)

fig.show()

In [50]:
membership_comparison = df.groupby("churned")["membership_years"].mean().reset_index()

membership_comparison["churned"] = membership_comparison["churned"].map({
    0: "Not Churned",
    1: "Churned"
})

fig = px.bar(
    membership_comparison,
    x="churned",
    y="membership_years",
    color="churned",
    text_auto=".2f",
    title="Average Membership Years by Churn Status",
    labels={
        "churned": "Customer Status",
        "membership_years": "Average Membership Years"
    }
)

fig.show()

In [51]:
country_churn = (
    df.groupby("country")["churned"]
    .mean()
    .mul(100)
    .round(2)
    .reset_index(name="churn_rate")
    .sort_values("churn_rate", ascending=False)
)

fig = px.bar(
    country_churn,
    x="country",
    y="churn_rate",
    color="churn_rate",
    text_auto=".2f",
    title="Customer Churn Rate by Country",
    labels={
        "country": "Country",
        "churn_rate": "Churn Rate (%)"
    }
)

fig.show()

In [52]:
df.dtypes

customer_id                   int64
age                           int64
gender                          str
country                         str
annual_income                 int64
spending_score                int64
num_purchases                 int64
avg_purchase_value          float64
membership_years              int64
website_visits_per_month      int64
cart_abandon_rate           float64
churned                       int64
feedback_text                   str
last_purchase_date              str
dtype: object

In [ ]:
df["last_purchase_date"] = pd.to_datetime(df["last_purchase_date"])



In [56]:
df["last_purchase_date"].dtypes

dtype('<M8[us]')

### Key Findings & Business Recommendations

Key Findings

Customer Churn: 10.9% of customers are classified as churned.

Purchasing Behavior: Churned customers have fewer purchases on average (17.53 vs. 23.19) and a higher cart abandonment rate (63% vs. 49%).

Customer Engagement: Churned customers have lower website visits, spending scores, and average purchase values. Their average membership duration is also shorter.

Business Recommendations

Improve Checkout Experience: Investigate potential barriers contributing to the higher cart abandonment rate.

Increase Customer Engagement: Explore targeted campaigns to encourage repeat purchases and improve customer retention.

Validate Key Drivers: Further investigate the observed patterns before treating them as causes of churn, particularly the relationship between feedback and churn status.

Conclusion

The analysis highlights behavioral differences between churned and non-churned customers. These findings can guide further investigation into customer retention, but they do not establish causality.